# MedGemma Playground

Interact directly with the MedGemma model — test loading, text generation, JSON output, and vision.

## 1. Setup & Auth

In [ ]:
# Pick your model — change this to whichever you have access to
MODEL_ID = "google/medgemma-4b-it"
# MODEL_ID = "google/medgemma-1.5-4b-it"

USE_4BIT = False  # only needed if VRAM < 16 GB
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.3

In [10]:
# Login to HuggingFace (needed for gated models)
# Option A: run this cell and paste your token
from huggingface_hub import login
login()

In [12]:
# Option B: if you already have HF_TOKEN in env, this should show True
import os
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")))

HF_TOKEN set: False


## 2. Load Model

In [13]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
VRAM: 85.0 GB


In [14]:
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=True)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print("Processor loaded OK")

Processor loaded OK


In [ ]:
if USE_4BIT and torch.cuda.is_available():
    from transformers import BitsAndBytesConfig
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        trust_remote_code=True,
    ).to(device)

# Sync vocab size in case tokenizer and model embeddings differ
model.resize_token_embeddings(len(processor.tokenizer))

print(f"Model loaded on {model.device}")

## 3. Helper: Generate Text

In [ ]:
def generate(prompt: str, max_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE) -> str:
    """Send a text prompt to the model and return the response."""
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    # Gemma does not use token_type_ids — passing them causes NaN logits
    inputs.pop("token_type_ids", None)

    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
        )
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    print(f"[{input_len} input tokens -> {outputs[0].shape[-1] - input_len} output tokens]")
    return response


def generate_with_image(image, prompt: str, max_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE) -> str:
    """Send an image + text prompt to the model and return the response."""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ]}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    # Gemma does not use token_type_ids — passing them causes NaN logits
    inputs.pop("token_type_ids", None)

    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
        )
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    print(f"[{input_len} input tokens -> {outputs[0].shape[-1] - input_len} output tokens]")
    return response

## 4. Basic Text Generation

In [ ]:
# Simple medical question
response = generate("What are the symptoms of malaria?")
print(response)

## 5. JSON Output (Pydantic Schema)

This is what HealthPost uses — prompt the model with a JSON schema and parse with Pydantic.

In [ ]:
import json
from pydantic import BaseModel, Field
from typing import List, Optional


class Medication(BaseModel):
    name: str
    dosage: str
    duration: Optional[str] = None


class ClinicalAssessment(BaseModel):
    condition: str
    confidence: str  # "high" / "medium" / "low"
    differential_diagnoses: List[str] = Field(default_factory=list)
    known_symptoms: List[str] = Field(default_factory=list)
    medications: List[Medication] = Field(default_factory=list)
    instructions: List[str] = Field(default_factory=list)
    warning_signs: List[str] = Field(default_factory=list)
    follow_up_days: int = 3
    requires_referral: bool = False
    referral_reason: Optional[str] = None


schema = ClinicalAssessment.model_json_schema()
print(json.dumps(schema, indent=2))

In [ ]:
# Build the diagnosis prompt
symptoms = "Patient has had high fever for 3 days, shaking chills especially at night, severe headache, body aches, and sweating. Lives in malaria-endemic area."
patient_age = "adult"
current_medications = ["Metformin", "Amlodipine"]

prompt = (
    "You are a medical decision support system for a Community Health Worker.\n"
    "Analyze the patient case and respond with ONLY a valid JSON object.\n\n"
    "PATIENT CASE:\n"
    f"Age: {patient_age}\n"
    f"Symptoms: {symptoms}\n"
    f"Current medications: {', '.join(current_medications)}\n"
    f"\nRespond with ONLY valid JSON matching this schema:\n"
    f"{json.dumps(schema, indent=2)}\n\n"
    "JSON response:"
)

print(f"Prompt length: {len(prompt)} chars")
print("---")
print(prompt[:500], "...")

In [ ]:
# Generate and see the raw response
import re

raw_response = generate(prompt, max_tokens=1024)
print("=== RAW RESPONSE ===")
print(raw_response)

In [ ]:
# Parse with Pydantic
def parse_json_response(response: str) -> ClinicalAssessment:
    """Extract JSON from LLM response, handle markdown fences."""
    cleaned = re.sub(r"```(?:json)?\s*", "", response)
    cleaned = cleaned.strip().rstrip("`")

    start = cleaned.find("{")
    end = cleaned.rfind("}") + 1
    if start >= 0 and end > start:
        json_str = cleaned[start:end]
        return ClinicalAssessment.model_validate_json(json_str)

    raise ValueError("No JSON object found in response")


try:
    assessment = parse_json_response(raw_response)
    print("=== PARSED ===")
    print(json.dumps(assessment.model_dump(), indent=2))
except Exception as e:
    print(f"Parse failed: {e}")
    print("You may need to tweak the prompt or try a different temperature.")

## 6. Vision (Image Analysis)

Load an image and ask the model to analyze it.

In [ ]:
from PIL import Image

# Load a test image — replace with your own path
# image = Image.open("path/to/medical_image.jpg")
# display(image)

# Or create a dummy image to test the pipeline works
image = Image.new("RGB", (224, 224), color=(200, 150, 150))
print(f"Image size: {image.size}")

In [ ]:
vision_response = generate_with_image(
    image,
    "Describe any medical findings visible in this image. "
    "List each finding on its own line.",
    max_tokens=256,
)
print(vision_response)

## 7. Free Experimentation

Try your own prompts below.

In [ ]:
# Try any prompt here
response = generate("What is the first-line treatment for uncomplicated malaria in sub-Saharan Africa?")
print(response)